# 상체 박스가 전신 박스 안에 들어간 경우의 원판 추적

입력 640과 배포 모델을 고정하고 원판 후보 필터만 비교했다. 기존 검증 6명의 미탐은 두 조건 모두 5장이었다. 작은 박스의 **90% 이상**이 큰 박스에 포함되고 작은 박스가 더 짧으며 그 중심이 위쪽 절반인 경우를 비교한다. 구간은 ready부터 finish 또는 drive까지다.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from IPython.display import display
from wlcv.config import ROOT, VIDEOS
from wlcv.data.frames import read_rgb

out = ROOT / "outputs/performer_containment/20261127"
if not (out / "by_video.csv").is_file():
    out = ROOT / "docs/results/performer_containment"
cache = ROOT / "outputs/foreground_thresholds/20261127-154950/candidates"
scores = pd.read_csv(out / "by_video.csv")
pairs = pd.read_csv(out / "pairs.csv")
reviews = pd.read_csv(ROOT / "data/annotations/performer_containment_review.csv")
keys = ["video_id", "frame_index", "small", "large"]
audit = pairs.merge(reviews, on=keys, how="outer", validate="one_to_one", indicator=True)
assert audit["_merge"].eq("both").all()

## 같은 정답으로 채점한 결과
정탐·미탐·오탐은 수동 정답 270장에서, 좌표 누락은 동작 전체 478프레임에서 센다.

In [ ]:
motion = scores.query("phase == 'motion'")
display(motion.groupby("condition").agg({"labeled": "sum", "tp": "sum", "fn": "sum", "fp": "sum", "frames": "sum", "missing": "sum", "longest": "max"}))
display(motion.pivot(index="video_id", columns="condition", values="fn"))

## 두 사람을 잘못 묶었는지 검수
표의 16장은 원판 결과가 변했는지와 무관하게 포함 기준으로 묶인 **모든 프레임**이다. 영상과 번호를 바꾸면 그 장면의 작은 박스와 큰 박스를 원본에서 확인할 수 있다. 선수 신원에 대한 판정은 원본 육안 검수에 따른 것이다.

In [ ]:
display(audit[["video_id", "frame_index", "small", "large", "coverage", "phase", "pool_changed", "plate_changed", "visual_verdict"]])
video = "b02_sn_h_02"
frame = 34
if (VIDEOS / f"{video}.mp4").is_file() and (cache / f"{video}.json").is_file():
    row = audit[(audit.video_id == video) & (audit.frame_index == frame)].iloc[0]
    people = json.loads((cache / f"{video}.json").read_text())[frame]["people"]
    fig, ax = plt.subplots(figsize=(4, 5))
    ax.imshow(read_rgb(VIDEOS / f"{video}.mp4", frame))
    for number, label, color in [(row.large, "large", "gold"), (row.small, "small", "deepskyblue")]:
        x1, y1, x2, y2 = people[number-1]
        ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False, color=color, lw=2, label=label))
    ax.legend()
    ax.set_title(f"{video} / {frame}")
    ax.axis("off")
    plt.show()
else:
    print("박스를 원본에 겹쳐 보려면 로컬 영상과 해당 영상의 후보 JSON이 필요합니다.")

## 가설을 제시한 54번
이 장면은 검증 6명 점수에 들어가지 않는다. 54번의 원판은 복구됐지만 자세는 보류됐다.

In [ ]:
case = pd.read_csv(out / "origin_case.csv")
display(case[["frame_index", "condition", "reason", "source", "confidence", "iou_54"]])

공개 CSV로 위 표를 재현할 수 있다. 원본과 기존 검증 후보 캐시가 준비된 환경에서는 저장소 루트에서 `python -m wlcv.evaluation.plate_person_containment --runtime . --case`로 첫 프레임부터 다시 비교한다. 평가에 고정한 배포 런타임은 리비전 `430ccb2`다.